<div style="max-width:100%;box-sizing:border-box;overflow:visible;border-top:4px solid #0f766e;padding:32px 0 20px;margin:0 0 24px">
  <div style="display:block;color:#0f766e;font-size:13px;line-height:1.8;font-weight:700;letter-spacing:0.8px;text-transform:uppercase;margin:0 0 8px">LAB 6 · DATA WAREHOUSING WITH APACHE DORIS</div>
  <div style="color:#17212b;font-size:30px;line-height:1.3;font-weight:750;margin:0 0 10px">乱序裁决、历史保留和可恢复重放</div>
  <p style="color:#475569;font-size:15px;line-height:1.7;max-width:900px;margin:0">使用统一订单样本，观察 SQL、结果与验收证据。请按顺序运行单元。</p>
  <span style="display:inline-block;border:1px solid #99f6e4;border-radius:4px;background:#f0fdfa;color:#115e59;padding:6px 10px;margin-top:14px;font-size:12px">Doris 4.1.3 target · Order data · Isolated course database</span>
</div>

By the end of this lab, you will have a running Doris environment, an `events` table containing more than 10 million rows, and analytical results produced from that table. Run the cells in order.

[讲义](course6_updates_deletes_and_replay.md) · [课程入口](../README.md)


## 前置条件

先完成 D09-A；本 Lab 必须读取 orders_clean_demo，不能绕过准入。只重建 d06_current、d06_history、d06_deliveries、d06_partial、d06_delete。本实验是显式业务事件重放，不是 Binlog CDC 恢复。


In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "dw_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from dw_course import WarehouseLab
from dw_course.runtime import COURSE_ROOT, fixture, expect, normalized
from dw_course.schema import ORDER_COLUMNS, order_ddl, order_rows
from dw_course.ui import show_sql, show_response

lab = WarehouseLab()




## 1. 分开建当前表与历史表

当前表 UNIQUE KEY(order_id) 使用 event_version 裁决；历史表 UNIQUE KEY(event_id) 保留不同业务事件。两者由实验代码显式维护，不假设跨表原子提交。


In [ ]:
lab.execute("DROP TABLE IF EXISTS d06_current")
ddl = order_ddl("d06_current", current=True)
show_sql("建表 SQL", ddl)
lab.execute(ddl)
lab.execute("DROP TABLE IF EXISTS d06_history")
ddl = order_ddl("d06_history", history=True)
show_sql("建表 SQL", ddl)
lab.execute(ddl)
lab.execute(f"INSERT INTO d06_current ({','.join(ORDER_COLUMNS)}) SELECT {','.join(ORDER_COLUMNS)} FROM orders_clean_demo")
lab.execute(f"INSERT INTO d06_history ({','.join(ORDER_COLUMNS)}) SELECT {','.join(ORDER_COLUMNS)} FROM orders_clean_demo")
expect(lab.query("SELECT COUNT(*), SUM(order_amount) FROM d06_current"), [(10,"1400.00")])
lab.execute("DROP TABLE IF EXISTS d06_deliveries")
lab.execute('CREATE TABLE d06_deliveries (attempt_id BIGINT, delivery_id BIGINT, event_id VARCHAR(32), payload STRING) DUPLICATE KEY(attempt_id, delivery_id) DISTRIBUTED BY HASH(attempt_id) BUCKETS 1 PROPERTIES("replication_num"="1")')


## 2. 模拟中断，再恢复

先记录第一条投递并写入历史，然后模拟在当前表写入之前中断。再次投递整批事件，保留每次投递，按稳定事件 ID 去重历史。该实验只中断课程步骤，不终止数据库进程。


In [ ]:
import json
deliveries = fixture("deliveries.json")
first = deliveries[0]
lab.insert("d06_deliveries", ["attempt_id","delivery_id","event_id","payload"],
           [(0, first["delivery_id"], first["event_id"], json.dumps(first))])
lab.insert("d06_history", ORDER_COLUMNS, order_rows([first]))
expect(lab.query("SELECT status FROM d06_current WHERE order_id=1001"), [("CREATED",)])

def replay(attempt):
    for record in deliveries:
        lab.insert("d06_deliveries", ["attempt_id","delivery_id","event_id","payload"],
                   [(attempt, record["delivery_id"], record["event_id"], json.dumps(record))])
        lab.insert("d06_history", ORDER_COLUMNS, order_rows([record]))
        lab.insert("d06_current", ORDER_COLUMNS, order_rows([record]))

replay(1)


## 3. 逐行对账，再完整重放一次

1001 必须保持版本 3 的 SHIPPED，1003 保持 REFUNDED 且支付金额仍为 150。原始投递数增加是正常现象，逻辑历史和当前状态不能变化。


In [ ]:
projection = ",".join("DATE_FORMAT(event_time, '%Y-%m-%d %H:%i:%s')" if col == "event_time" else col for col in ORDER_COLUMNS)
def verify_state():
    expect(lab.query(f"SELECT {projection} FROM d06_current ORDER BY order_id"), order_rows(fixture("expected_current.json")))
    expect(lab.query("SELECT COUNT(*), SUM(order_amount), SUM(paid_amount), SUM(refund_amount) FROM d06_current"),
           [(11,"1510.00","250.00","150.00")])
    expect(lab.query("SELECT COUNT(*) FROM d06_history"), [(16,)])
    expected_history = {r["event_id"]:r for r in fixture("orders.json") + deliveries}
    expect(lab.query(f"SELECT {projection} FROM d06_history ORDER BY event_id"),
           order_rows([expected_history[key] for key in sorted(expected_history)]))
verify_state()
replay(2)
verify_state()
expect(lab.query("SELECT COUNT(*) FROM d06_deliveries"), [(15,)])


## 4. 部分列更新

仅在独立表上测试；更新版本也要递增。实验记录当前会话的部分更新开关，并在 finally 中恢复。


In [ ]:
lab.execute("DROP TABLE IF EXISTS d06_partial")
ddl = order_ddl("d06_partial", current=True)
show_sql("建表 SQL", ddl)
lab.execute(ddl)
lab.insert("d06_partial", ORDER_COLUMNS, order_rows(fixture("orders.json")[:1]))
previous_partial = lab.query("SELECT @@enable_unique_key_partial_update")[0][0]
try:
    lab.execute("SET enable_unique_key_partial_update = true")
    lab.execute("INSERT INTO d06_partial (order_id, status, event_version) VALUES (1001, 'CANCELLED', 2)")
finally:
    lab.execute("SET enable_unique_key_partial_update = %s", (previous_partial,))
expect(lab.query("SELECT status, event_version, order_amount, region FROM d06_partial"),
       [("CANCELLED",2,"100.00","EAST")])


## 5. 独立删除实验

主线退款订单不删除。副本中用 is_deleted 表达业务软删除，再用 SQL DELETE 删除另一行；查询不可见不等于磁盘立即回收。


In [ ]:
lab.execute("DROP TABLE IF EXISTS d06_delete")
lab.execute('CREATE TABLE d06_delete (order_id BIGINT, is_deleted BOOLEAN, status VARCHAR(20)) UNIQUE KEY(order_id) DISTRIBUTED BY HASH(order_id) BUCKETS 1 PROPERTIES("replication_num"="1", "enable_unique_key_merge_on_write"="true")')
lab.insert("d06_delete", ["order_id","is_deleted","status"], [(1001,False,"CREATED"),(1002,False,"CREATED")])
lab.execute("UPDATE d06_delete SET is_deleted=true WHERE order_id=1001")
expect(lab.query("SELECT COUNT(*) FROM d06_delete"), [(2,)])
expect(lab.query("SELECT order_id FROM d06_delete WHERE is_deleted=false"), [(1002,)])
lab.execute("DELETE FROM d06_delete WHERE order_id=1002")
expect(lab.query("SELECT order_id FROM d06_delete"), [(1001,)])
verify_state()
lab.close()


## 完成与剩余边界

订单行、金额、历史与重复重放都要通过。导入删除标记、真实 CDC 快照切换与位点恢复、事件 ID 冲突策略尚需后续集成验证；不能将这些结论从本地人工事件自动外推。
